In [9]:
import pandas as pd
import re
import unicodedata
from langdetect import detect, DetectorFactory
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

# Đảm bảo kết quả nhận diện ngôn ngữ ổn định
DetectorFactory.seed = 0

# 1. Từ điển viết tắt (Bạn có thể thêm các từ khác vào đây)
abbreviation_dict = {
    r'\bdc\b': 'được', r'\bđc\b': 'được', r'\bđk\b': 'được',
    r'\bko\b': 'không', r'\bk\b': 'không', r'\bkg\b': 'không', r'\bhk\b': 'không', r'\bhong\b': 'không', r'\bkhg\b': 'không',
    r'\bsp\b': 'sản phẩm',
    r'\bng\b': 'người',
    r'\bmn\b': 'mọi người', r'\bmng\b': 'mọi người',
    r'\bh\b': 'giờ',
    r'\bkh\b': 'khách',
    r'\bgw\b': 'giao',
    r'\bnt\b': 'nhắn tin',
    r'\bshop\b': 'cửa hàng', r'\bsop\b': 'cửa hàng', r'\bshops\b': 'cửa hàng',
    r'\btks\b': 'cảm ơn', r'\bthanks\b': 'cảm ơn', r'\bthank\b': 'cảm ơn',
    r'\bck\b': 'chồng',
    r'\bbt\b': 'bình thường',
    r'\bmik\b': 'mình',
    r'\blm\b': 'làm',
    r'\bcskh\b': 'chăm sóc khách hàng',
    r'\btphcm\b': 'thành phố hồ chí minh',
    r'\bvãi\b': 'vải',
    r'\btrc\b': 'trước',
    r'\bnv\b': 'nhân viên',
    r'\bvs\b': 'với',
    r'\bfrom\b': 'form', r'\bfom\b': 'form', r'\bphom\b': 'form',
    r'\bsezi\b': 'size',
    r'\bqc\b': 'quảng cáo',
    r'\bcx\b': 'cũng',
    r'\bib\b': 'nhắn tin'
}

def is_vietnamese(text):
    """Kiểm tra xem câu có phải tiếng Việt hay không"""
    if not text or len(text) < 5: # Câu quá ngắn khó nhận diện chính xác
        return True 
    try:
        # Nếu nhận diện là tiếng Anh ('en') thì trả về False
        return detect(text) != 'en'
    except:
        return True

def parse_relative_date(date_str, ref_date_str="2025-01-18"):
    """
    Chuyển đổi chuỗi "x ngày/tuần/tháng trước" thành định dạng YYYY-MM-DD.
    """
    if pd.isna(date_str) or str(date_str).strip() == "":
        return date_str
    
    date_str = str(date_str).lower().strip()
    
    # 1. Nếu đã đúng định dạng YYYY-MM-DD (4 số - 2 số - 2 số) thì trả về luôn
    if re.match(r'\d{4}-\d{2}-\d{2}', date_str):
        return date_str
    
    # 2. Thiết lập ngày mốc (Reference Date)
    ref_date = datetime.strptime(ref_date_str, "%Y-%m-%d")
    
    # 3. Dùng Regex để tìm số và đơn vị thời gian
    # Tìm mẫu: "số + đơn vị"
    match = re.search(r'(\d+)\s+(ngày|tuần|tháng|năm)', date_str)
    
    if match:
        value = int(match.group(1))
        unit = match.group(2)
        
        if 'ngày' in unit:
            result_date = ref_date - timedelta(days=value)
        elif 'tuần' in unit:
            result_date = ref_date - timedelta(weeks=value)
        elif 'tháng' in unit:
            # Dùng relativedelta để trừ tháng chính xác (ví dụ trừ 1 tháng từ 31/03 ra 28/02)
            result_date = ref_date - relativedelta(months=value)
        elif 'năm' in unit:
            result_date = ref_date - relativedelta(years=value)
        else:
            return date_str
            
        return result_date.strftime('%Y-%m-%d')
    
    # Xử lý trường hợp đặc biệt như "hôm qua" nếu có
    if 'hôm qua' in date_str:
        return (ref_date - timedelta(days=1)).strftime('%Y-%m-%d')
        
    return date_str

def clean_and_normalize(text):
    if pd.isna(text) or str(text).strip() == "":
        return None # Trả về None để lát nữa xoá rỗng dễ dàng
    
    text = str(text)
    
    # A. Xử lý xuống dòng và Unicode
    text = re.sub(r'[\n\r\t]', ' ', text)
    text = unicodedata.normalize('NFC', text).lower()
    
    # B. Xóa các câu thuần tiếng Anh
    if not is_vietnamese(text):
        return None

    # C. Chuyển đổi từ viết tắt (Sử dụng Regex word boundary \b)
    for abbr, full in abbreviation_dict.items():
        text = re.sub(abbr, full, text)
        
    # D. Làm sạch ký tự đặc biệt và ký tự lặp
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    text = re.sub(r'[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ.,!]', ' ', text)
    
    # E. Xóa từ rác quá dài (gibberish)
    words = [w for w in text.split() if len(w) <= 20]
    text = " ".join(words)
    
    final_text = re.sub(r'\s+', ' ', text).strip()
    return final_text if final_text != "" else None

# --- THỰC THI ---
file_path = 'lazada_reviews_SP05_1801.xlsx'

# Đọc file (Hỗ trợ cả Excel và CSV)
if file_path.endswith('.xlsx'):
    df = pd.read_excel(file_path)
else:
    df = pd.read_csv(file_path, encoding='utf-8-sig')

print("⏳ Đang xử lý làm sạch và lọc ngôn ngữ...")

# 1. Áp dụng hàm làm sạch
df['Nội dung review'] = df['Nội dung review'].apply(clean_and_normalize)

# 2. XOÁ RỖNG (Xoá các dòng đã bị lọc là tiếng Anh hoặc rỗng ban đầu)
df = df.dropna(subset=['Nội dung review'])

# 3. Loại bỏ câu đánh giá ngắn (Nhỏ hơn hoặc bằng 5 từ)
df = df[df['Nội dung review'].apply(lambda x: len(str(x).split()) > 5)]

# 4. Xoá trùng lặp câu
count_before = len(df)
df = df.drop_duplicates(subset=['Nội dung review'], keep='first')
count_after = len(df)

print(f"🗑️ Đã loại bỏ {count_before - count_after} dòng trùng lặp.")

# 5. Thay đổi ngày/tuần/tháng trước theo đúng định dạng ngày
reference_day = "2025-01-18"
df['Ngày review'] = df['Ngày review'].apply(lambda x: parse_relative_date(x, reference_day))

# 5. Đánh lại Số thứ tự (STT)
if 'STT' in df.columns:
    df['STT'] = range(1, len(df) + 1)

# 6. Ghi đè lại file
if file_path.endswith('.xlsx'):
    df.to_excel(file_path, index=False, engine='openpyxl')
else:
    df.to_csv(file_path, index=False, encoding='utf-8-sig')

print(f"🚀 Đã xử lý file {file_path} thành công. Còn lại {len(df)} dòng.")

⏳ Đang xử lý làm sạch và lọc ngôn ngữ...
🗑️ Đã loại bỏ 55 dòng trùng lặp.
🚀 Đã xử lý file lazada_reviews_SP05_1801.xlsx thành công. Còn lại 291 dòng.
